## CONFIGURACIÓN DEL CUADERNO

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from keras import layers

# Macros

PATH_ORI = '/content/drive/MyDrive/TFG/dt_ori'
PATH_AUG = '/content/drive/MyDrive/TFG/dt_aug'

PATH_TRAIN_ORI = '/content/drive/MyDrive/TFG/dt_aug/train'
PATH_TRAIN = '/content/drive/MyDrive/TFG/dataset/train'
PATH_VAL_ORI = '/content/drive/MyDrive/TFG/dt_ori/val'
PATH_VAL = '/content/drive/MyDrive/TFG/dataset/val'
PATH_TEST_ORI = '/content/drive/MyDrive/TFG/dt_ori/test'
PATH_TEST = '/content/drive/MyDrive/TFG/dataset/test'

LABELS = {'0':'fluido', '1':'moderado', '2':'denso', '3':'saturado'}


IMAGE_SIZE = (540, 960)


## Equilibrado del dataset

Se aplica aumentación selectiva para equilibrar las clases dentro de cada semáforo y condición de iluminación (día/noche). El objetivo por clase se calcula como el doble de la clase mayoritaria global (`max_n_label`), con un factor máximo de ×10 aumentaciones por imagen original.

Los semáforos con varias condiciones de iluminación usan `max_n_label / 2` como objetivo local para evitar que el total del semáforo supere a semáforos sin esa distinción.

Los casos con menos de 5 originales en alguna clase no alcanzan el equilibrio completo y se tratan de igualar a la clase interna mayoritaria.

In [ ]:
MAX_ITERA = 10 #número máximo de aumentaciones sobre una imagen


#Funciones para el aumento de datos selectivo:

# detección de imágenes originales para evitar modificar imágenes aumentadas
def original(nombre_img):
  return 'aug' not in nombre_img

#detección de condiciones atmosféricas dentro del semáforo
def get_condiciones(path_sem):

  contenido_sem = os.listdir(path_sem)
  condiciones = []
  if 'dia' in contenido_sem : condiciones.append ('dia')
  if 'noche' in contenido_sem : condiciones.append ('noche')
  if len(condiciones) == 0 : condiciones = [None]

  return condiciones

#obtine el número de muestras de la clase mayoritaria del dataset
def get_max_label(path):

  max_n_label = 0
  flag_cond = False
  for sem in sorted(os.listdir(path)):
    path_sem = os.path.join(path, sem)
    for cond in get_condiciones(path_sem):
      path_cond = os.path.join(path_sem, cond) if cond else path_sem
      for label in LABELS:
        path_label = os.path.join(path_cond, label)
        n = len(os.listdir(path_label))
        if n > max_n_label :
          max_n_label = n
          path_max = path_label
          if cond : flag_cond = True
          else    : flag_cond = False

  print (f"La clase mayoritaria se encuentra en {path_max} y contiene {max_n_label} imágenes")
  if flag_cond : max_n_label = max_n_label*2

  return max_n_label


#cuenta el número de imágenes dentro de cada semáforo o condición
def recuento_labels (path) :

  n_label = []
  for label in LABELS:
    path_label = os.path.join(path, label)
    if os.path.isdir(path_label):
      n = 0
      for img in os.listdir(path_label):
        if original(img) : n += 1
    else : n = 0

    n_label.append(n)

  return n_label

# Función que calcula las aumentaciones óptimas para lograr el equilibrio
def calculo_it(path, max_n_label):

  n_label = recuento_labels(path)
  if (max(n_label)/min(n_label)>10) : max_n_label = max(n_label)
  num_itera = []
  for n in n_label:
    if n == max_n_label : num_itera.append(1)
    else :
      num_itera_i = round(max_n_label/n)
      if num_itera_i < 1 : num_itera_i = 1
      if num_itera_i > MAX_ITERA : num_itera_i = MAX_ITERA
      num_itera.append(num_itera_i)

  return num_itera

# aumentación por clase según número de veces
def aug_label(ori_dir, aug_dir, n_veces):

  os.makedirs(aug_dir, exist_ok=True)

  ficheros = sorted([
    f for f in os.listdir(ori_dir)
    if f.lower().endswith(('.jpg', '.png', '.jpeg')) and original(f)
  ])

  for f in ficheros:
    img = Image.open(os.path.join(ori_dir, f)).convert("RGB")
    img.resize(IMAGE_SIZE, Image.LANCZOS).save(os.path.join(aug_dir, f))

  contador = 0
  for f in ficheros:
    nombre_base = os.path.splitext(f)[0]
    img_orig = np.array(
        Image.open(os.path.join(ori_dir, f)).convert("RGB"),
        dtype=np.float32
    ) / 255.0

    img_tensor = tf.expand_dims(img_orig, axis=0)

    for i in range(n_veces-1):
      img_aug = data_augmentation(img_tensor, training=True).numpy()[0]
      img_aug = (np.clip(img_aug, 0, 1) * 255).astype(np.uint8)

      nombre_aug = f"{nombre_base}_aug{contador:04d}.jpg"
      Image.fromarray(img_aug)\
            .resize(IMAGE_SIZE, Image.LANCZOS)\
            .save(os.path.join(aug_dir, nombre_aug), quality=92)
      contador += 1

  return contador

#Bloque de transformaciones para realizar aumento de datos en el dataset de entrenamiento

data_augmentation = keras.Sequential([
  #variaciones geométricas
  layers.RandomRotation(factor=(-0.013, 0.015)),
  layers.RandomZoom(0.1),
  layers.RandomTranslation(
      height_factor=0.05,
      width_factor=(0, 0.05),
      fill_mode='reflect'
  ),
  #variaciones fotométricas
  layers.RandomBrightness(factor=0.2, value_range=[0.0, 1.0]),
  layers.RandomContrast(factor=0.2),
  layers.GaussianNoise(0.02),

  layers.Lambda(lambda x: tf.clip_by_value(x, 0.0, 1.0))
])

def augmentation (path_data, path_aug) :

  if os.path.isdir(path_aug):
    raise FileExistsError(f"Ya existe el fichero de augmentation")

  max_n_label = get_max_label(path_data)
  for semaforo in sorted(os.listdir(path_data)):
    path_sem = os.path.join(path_data, semaforo)
    for cond in get_condiciones(path_sem):
      if cond is not None:
        path_cond = os.path.join(path_sem, cond)
        num_it_label = calculo_it(path_cond, max_n_label/2)
      else:
        path_cond = path_sem
        num_it_label = calculo_it(path_cond, max_n_label)

      n_label = recuento_labels(path_cond)

      for label in LABELS :
        n_orig = n_label[int(label)]
        n_veces = num_it_label[int(label)]
        path_ori = os.path.join(path_cond, label)

        if cond is not None:
          path_fin = os.path.join(path_aug, semaforo, cond, label)
        else:
          path_fin = os.path.join(path_aug, semaforo, label)

        n_generadas = aug_label(path_ori, path_fin, n_veces)

        cond_str    = cond if cond is not None else "—"
        print(f"[{semaforo}][{cond_str}][{label}]: {n_generadas} aug ")


In [ ]:
augmentation (PATH_ORI, PATH_AUG)

## Reorganización del dataset
El código que se define a continuación busca reorganizar el dataset con una estructura de `dataset/semaforo/[condición/]etiqueta` a una estructura de `dataset/etiqueta`.

In [ ]:
import os
import shutil
from pathlib import Path


def copiar(origen, clase, path_dest, resumen):
  destino = path_dest / clase
  destino.mkdir(parents=True, exist_ok=True)

  ficheros = []
  for f in origen.iterdir():
    if f.suffix.lower() == '.jpg':
      ficheros.append(f)

  for f in ficheros:
    shutil.copy2(f, destino / f.name)
  resumen[clase] = resumen.get(clase, 0) + len(ficheros)

def reorganizar_dataset(path_ori, path_dest):

  path_ori  = Path(path_ori)
  path_dest = Path(path_dest)

  if not path_ori.exists():
    print(f"[ERROR] No existe el directorio origen: {path_ori}")
    return

  resumen = {}

  for sem in sorted(path_ori.iterdir()):
    if not sem.is_dir() : continue

    semaforo   = sem.name
    condiciones = get_condiciones(sem)

    for cond in condiciones:
      path_cond = sem / cond if cond else sem
      prefijo   = f"{semaforo}_{cond}"  if cond else semaforo

      for label in sorted(path_cond.iterdir()):
        if not label.is_dir() : continue
        copiar(label, label.name, path_dest, resumen)

  #Resultados
  print("Resultados de la copia: \n")
  for clase, n in sorted(resumen.items()) :
    print(f"  {clase}: {n} imágenes")
  print(f"  TOTAL: {sum(resumen.values())} imágenes")


In [ ]:
reorganizar_dataset(
    path_ori = PATH_TRAIN_ORI,
    path_dest = PATH_TRAIN
)

reorganizar_dataset(
    path_ori = PATH_VAL_ORI,
    path_dest = PATH_VAL
)

reorganizar_dataset(
    path_ori = PATH_TEST_ORI,
    path_dest = PATH_TEST
)

##RESUMEN DE LAS IMÁGENES QUE COMPONEN EL DATASET

In [ ]:
def resumen_dataset (path) :
  dataset = {
      'train': {'0':0, '1':0, '2':0, '3':0},
      'val': {'0':0, '1':0, '2':0, '3':0},
      'test': {'0':0, '1':0, '2':0, '3':0}
  }

  for sub_dataset in sorted(os.listdir(path)):
    path_sub = os.path.join(path, sub_dataset)
    if os.path.isdir(path_sub):
      n = 0
      for label in os.listdir(path_sub):
        path_label = os.path.join(path_sub, label)
        if os.path.isdir(path_label):
          n = len(os.listdir(path_label))
          dataset[sub_dataset][label] = n

  print (f"Train: {dataset['train']}")
  print (f"Val: {dataset['val']}")
  print (f"Test: {dataset['test']}")

In [ ]:
resumen_dataset ("/content/drive/MyDrive/TFG/dataset")